# Análisis de `resultados_ap.csv`

Este cuaderno propone un análisis del comportamiento de los métodos disponibles sobre el problema de asignación (`AP`). Sigue la misma estructura del análisis de KP, pero adaptando las variables al esquema de AP: `tamano_matriz`, `coste_total`, `violacion`, `desviacion_relativa_optimo` y `penalizacion`.

#### Alcance del análisis

En AP solo se analizan las instancias `2x2` y `3x3`, porque las instancias `4x4` no se pudieron completar correctamente al producir error. El CSV principal contiene únicamente filas que encajan con la formulación AP real (`num_variables_qubo = num_filas * num_columnas`). Las filas antiguas que estaban en otro esquema se han conservado aparte en `resultados_ap_fuera_esquema.csv`, pero no se mezclan con este análisis.

#### Hipótesis de trabajo

Dado que en las ejecuciones disponibles las soluciones finales suelen alcanzar el óptimo (`ratio_optimo = 1`, `brecha_optimo = 0`, `violacion = 0`), el valor comparativo principal está en:

- Diferencias en el tiempo total y su descomposición.
- Probabilidad de éxito y calidad media de las muestras.
- Estabilidad entre ejecuciones.
- Escalabilidad entre `2x2` y `3x3`.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from matplotlib.ticker import MaxNLocator

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


In [ ]:
# Carga y limpieza específica de AP.
# El CSV principal contiene solo filas compatibles con el esquema AP actual.

ruta_csv = 'resultados_ap.csv'
df_completo = pd.read_csv(ruta_csv, na_values=['N/A'], dtype=str)

# Por si en el futuro se vuelve a pegar una cabecera dentro del CSV.
df_completo = df_completo[df_completo['id'] != 'id'].copy()

columnas_texto = {'tamano_matriz', 'metodo', 'warmstart'}
columnas_bool = {'factible', 'coincide'}

for col in df_completo.columns:
    if col in columnas_bool:
        df_completo[col] = df_completo[col].map({'True': True, 'False': False, True: True, False: False})
    elif col not in columnas_texto:
        df_completo[col] = pd.to_numeric(df_completo[col], errors='coerce')

tamanos_validos = ['2x2', '3x3']
df = df_completo[df_completo['tamano_matriz'].isin(tamanos_validos)].copy()

orden_metodos = [m for m in ['sa', 'qaoa', 'qaoa_warmstart'] if m in df['metodo'].dropna().unique()]
df['metodo'] = pd.Categorical(df['metodo'], categories=orden_metodos, ordered=True)
df_completo['metodo'] = pd.Categorical(df_completo['metodo'], categories=['sa', 'qaoa', 'qaoa_warmstart'], ordered=True)

fuera_alcance = len(df_completo) - len(df)
print('Dimensiones CSV completo:', df_completo.shape)
print('Dimensiones analizadas 2x2/3x3:', df.shape)
print('Filas conservadas fuera del análisis principal:', fuera_alcance)
print('Tamaños en CSV completo:', sorted(df_completo['tamano_matriz'].dropna().unique()))
print('Tamaños analizados:', sorted(df['tamano_matriz'].dropna().unique()))
display(df.head())


In [ ]:
print('Frecuencia por método en el CSV completo:')
display(df_completo['metodo'].value_counts(dropna=False).rename_axis('metodo').to_frame('frecuencia'))

print('Frecuencia por método en el análisis 2x2/3x3:')
display(df['metodo'].value_counts(dropna=False).rename_axis('metodo').to_frame('frecuencia'))

display(df['tamano_matriz'].value_counts().sort_index().rename_axis('tamano_matriz').to_frame('frecuencia'))


In [ ]:
variables_resumen = [
    'ratio_optimo', 'brecha_optimo', 'desviacion_relativa_optimo', 'violacion',
    'tiempo_total', 'prob_optimo', 'ratio_optimo_medio',
    't_solver', 't_cuantico', 't_clasico',
    'prob_optimo_muestras', 'prob_optimo_starts', 'tasa_factibilidad',
    'ratio_medio_factibles', 'coste_medio_start', 'std_coste_start',
    'tiempo_medio_start', 'penalizacion'
]
variables_resumen = [c for c in variables_resumen if c in df.columns]

tabla_descriptivos = df.groupby('metodo', observed=True)[variables_resumen].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
display(tabla_descriptivos)


### Análisis del tiempo total de ejecución

Se analiza primero la distribución del tiempo total por método. Como en KP, se muestran gráficos individuales por método y después una comparación agregada. En el CSV AP válido actual solo hay resultados de `sa` para `2x2` y `3x3`; las celdas siguen preparadas para incorporar `qaoa` y `qaoa_warmstart` cuando existan ejecuciones AP con el esquema correcto.


In [ ]:
metodos = list(df['metodo'].dropna().unique())

fig, axes = plt.subplots(1, len(metodos), figsize=(6 * max(len(metodos), 1), 6))
if len(metodos) == 1:
    axes = [axes]

for i, metodo in enumerate(metodos):
    sns.boxplot(
        data=df[df['metodo'] == metodo],
        y='tiempo_total',
        ax=axes[i],
        showfliers=True
    )
    axes[i].set_title(f'Tiempo total - {metodo}')
    axes[i].set_ylabel('Tiempo total (s)')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='metodo', y='tiempo_total', showfliers=True)
plt.title('Comparación del tiempo total por método')
plt.xlabel('Método')
plt.ylabel('Tiempo total (s)')
plt.tight_layout()
plt.show()


En las filas AP válidas actualmente disponibles, SA alcanza siempre soluciones factibles y óptimas, por lo que la comparación principal se desplaza hacia el coste temporal y la estabilidad del muestreo. El tiempo total aumenta claramente al pasar de `2x2` a `3x3`, aunque sigue manteniéndose en una escala muy baja para el enfoque clásico.

#### Escalabilidad con el tamaño del problema

Se representa la mediana del tiempo total en función del tamaño de la matriz y del número de variables QUBO. Dado que `4x4` queda fuera por error, esta sección compara únicamente `2x2` y `3x3`.


In [2]:
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import seaborn as sns

metodos = df['metodo'].unique()

# GRÁFICOS PARA CADA MÉTODO
fig, axes = plt.subplots(2, len(metodos), figsize=(18, 10))

# Escalabilidad con tamaño de matriz AP
for i, metodo in enumerate(metodos):
    df_metodo = df[df['metodo'] == metodo]

    sns.lineplot(
        data=df_metodo,
        x='tamano_matriz',
        y='tiempo_total',
        estimator='median',
        marker='o',
        ax=axes[0, i]
    )

    axes[0, i].set_title(
        f'Tamaño AP ({metodo})',
        fontweight='bold'
    )
    axes[0, i].set_xlabel('Tamaño de matriz')
    axes[0, i].set_ylabel('Tiempo total (s)')
    axes[0, i].xaxis.set_major_locator(MaxNLocator(integer=True))


# Escalabilidad con tamaño del QUBO
for i, metodo in enumerate(metodos):
    df_metodo = df[df['metodo'] == metodo]

    sns.lineplot(
        data=df_metodo,
        x='num_variables_qubo',
        y='tiempo_total',
        estimator='median',
        marker='o',
        ax=axes[1, i]
    )

    axes[1, i].set_title(
        f'Tamaño QUBO ({metodo})',
        fontweight='bold'
    )
    axes[1, i].set_xlabel('Número de variables QUBO')
    axes[1, i].set_ylabel('Tiempo total (s)')
    axes[1, i].xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.show()


# GRÁFICOS AGREGADOS
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.lineplot(
    data=df,
    x='tamano_matriz',
    y='tiempo_total',
    hue='metodo',
    estimator='median',
    marker='o',
    ax=axes[0]
)

axes[0].set_title(
    'Mediana del tiempo total según tamaño AP',
    fontweight='bold'
)
axes[0].set_xlabel('Tamaño de matriz')
axes[0].set_ylabel('Tiempo total (s)')
axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))


sns.lineplot(
    data=df,
    x='num_variables_qubo',
    y='tiempo_total',
    hue='metodo',
    estimator='median',
    marker='o',
    ax=axes[1]
)

axes[1].set_title(
    'Mediana del tiempo total según tamaño QUBO',
    fontweight='bold'
)
axes[1].set_xlabel('Número de variables QUBO')
axes[1].set_ylabel('Tiempo total (s)')
axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

El incremento entre `2x2` y `3x3` refleja el crecimiento esperado al aumentar el número de variables binarias de la formulación QUBO. En AP, este crecimiento es especialmente relevante porque el tamaño natural de la formulación crece con la matriz de asignación. Al no disponer de `4x4`, no se puede extrapolar una tendencia robusta más allá del rango pequeño analizado.

## Descomposición temporal de QAOA y QAOA warm-start

Esta sección replica el análisis de KP para los métodos basados en QAOA. Si el CSV limpio no contiene resultados AP válidos de `qaoa` o `qaoa_warmstart`, la celda informa de ello y no dibuja gráficos vacíos.


In [ ]:
df_q = df[df['metodo'].isin(['qaoa', 'qaoa_warmstart'])].copy()
componentes = ['t_cuantico', 't_clasico']

if df_q.empty or not df_q[componentes].notna().any().any():
    print('No hay resultados AP válidos de QAOA/QAOA warm-start para descomposición temporal.')
else:
    comp = (
        df_q.groupby('metodo', observed=True)[componentes]
        .median()
        .reset_index()
        .melt(id_vars='metodo', var_name='componente', value_name='mediana_tiempo')
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(data=comp, x='metodo', y='mediana_tiempo', hue='componente')
    plt.title('Descomposición mediana del tiempo del solver', fontweight='bold')
    plt.xlabel('Método')
    plt.ylabel('Tiempo mediano (s)')
    plt.tight_layout()
    plt.show()


In [ ]:
if df_q.empty or not df_q[componentes].notna().any().any():
    print('No hay resultados AP válidos de QAOA/QAOA warm-start para escalabilidad de componentes.')
else:
    comp_tamano = (
        df_q.groupby(['metodo', 'tamano_matriz'], observed=True)[componentes]
        .median()
        .reset_index()
        .melt(id_vars=['metodo', 'tamano_matriz'], var_name='componente', value_name='mediana_tiempo')
    )

    comp_qubo = (
        df_q.groupby(['metodo', 'num_variables_qubo'], observed=True)[componentes]
        .median()
        .reset_index()
        .melt(id_vars=['metodo', 'num_variables_qubo'], var_name='componente', value_name='mediana_tiempo')
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    sns.lineplot(data=comp_tamano, x='tamano_matriz', y='mediana_tiempo', hue='componente', style='metodo', markers=True, dashes=False, ax=axes[0])
    axes[0].set_title('Tiempo mediano por tamaño AP', fontweight='bold')
    axes[0].set_xlabel('Tamaño de matriz')
    axes[0].set_ylabel('Tiempo mediano (s)')

    sns.lineplot(data=comp_qubo, x='num_variables_qubo', y='mediana_tiempo', hue='componente', style='metodo', markers=True, dashes=False, ax=axes[1])
    axes[1].set_title('Tiempo mediano por tamaño QUBO', fontweight='bold')
    axes[1].set_xlabel('Número de variables QUBO')
    axes[1].set_ylabel('Tiempo mediano (s)')
    axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))

    plt.tight_layout()
    plt.show()


#### Métricas internas específicas por familia de método

Para QAOA se analizan `prob_optimo` y `ratio_optimo_medio`. Para SA se analizan `prob_optimo_muestras`, `prob_optimo_starts`, `tasa_factibilidad` y `ratio_medio_factibles`. Estas métricas permiten distinguir entre encontrar una buena solución final y generar muestras de buena calidad durante el proceso.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

if 'prob_optimo' in df.columns and df['prob_optimo'].notna().any():
    sns.boxplot(data=df[df['metodo'].isin(['qaoa', 'qaoa_warmstart'])], x='metodo', y='prob_optimo', ax=axes[0, 0])
    axes[0, 0].set_title('Probabilidad de óptimo en QAOA', fontweight='bold')
else:
    axes[0, 0].axis('off')
    axes[0, 0].set_title('Sin datos QAOA válidos')

if 'ratio_optimo_medio' in df.columns and df['ratio_optimo_medio'].notna().any():
    sns.boxplot(data=df[df['metodo'].isin(['qaoa', 'qaoa_warmstart'])], x='metodo', y='ratio_optimo_medio', ax=axes[0, 1])
    axes[0, 1].set_title('Ratio óptimo medio en QAOA', fontweight='bold')
else:
    axes[0, 1].axis('off')
    axes[0, 1].set_title('Sin datos QAOA válidos')

if 'prob_optimo_muestras' in df.columns and df['prob_optimo_muestras'].notna().any():
    sns.boxplot(data=df[df['metodo'] == 'sa'], x='tamano_matriz', y='prob_optimo_muestras', ax=axes[1, 0])
    axes[1, 0].set_title('Probabilidad de óptimo en muestras (SA)', fontweight='bold')
    axes[1, 0].set_xlabel('Tamaño AP')
else:
    axes[1, 0].axis('off')

if 'tasa_factibilidad' in df.columns and df['tasa_factibilidad'].notna().any():
    sns.boxplot(data=df[df['metodo'] == 'sa'], x='tamano_matriz', y='tasa_factibilidad', ax=axes[1, 1])
    axes[1, 1].set_title('Tasa de factibilidad (SA)', fontweight='bold')
    axes[1, 1].set_xlabel('Tamaño AP')
else:
    axes[1, 1].axis('off')

plt.tight_layout()
plt.show()


En SA, la solución final es óptima y factible en las instancias `2x2` y `3x3` disponibles, pero las métricas de muestreo muestran diferencias internas: la probabilidad de que una muestra individual sea óptima baja al pasar a `3x3`, y el `ratio_medio_factibles` también tiende a reducirse. Esto indica que el método sigue encontrando el óptimo final, pero el espacio de soluciones se vuelve más exigente incluso en este salto pequeño de tamaño.

## Contraste estadístico

El contraste apareado entre QAOA y QAOA *warm-start* se calcula solo si existen ambos métodos con instancias comparables dentro de `2x2` y `3x3`.


In [ ]:
columnas_base = ['id', 'tamano_matriz', 'num_variables_qubo', 'optimo']
df['instancia_clave'] = df[columnas_base].astype(str).agg(' | '.join, axis=1)

def cohens_d_paired(x, y):
    diff = np.array(x) - np.array(y)
    sd = diff.std(ddof=1)
    return diff.mean() / sd if sd != 0 else np.nan

comparables = df[df['metodo'].isin(['qaoa', 'qaoa_warmstart'])].copy()

if comparables['metodo'].nunique() < 2:
    print('No hay pares comparables qaoa vs qaoa_warmstart en AP 2x2/3x3.')
else:
    agg = comparables.groupby(['instancia_clave', 'metodo'], as_index=False, observed=True).agg({
        'tiempo_total': 'median',
        'prob_optimo': 'median',
        'ratio_optimo_medio': 'median',
        't_cuantico': 'median',
        't_clasico': 'median'
    })

    resultados = []
    for nombre in ['tiempo_total', 'prob_optimo', 'ratio_optimo_medio']:
        tabla = agg.pivot(index='instancia_clave', columns='metodo', values=nombre).dropna()
        if not {'qaoa', 'qaoa_warmstart'}.issubset(tabla.columns) or len(tabla) < 2:
            continue
        x = tabla['qaoa']
        y = tabla['qaoa_warmstart']
        diff = x - y
        shapiro_p = stats.shapiro(diff).pvalue if len(diff) >= 3 else np.nan
        if pd.notna(shapiro_p) and shapiro_p > 0.05:
            test_name = 't-test apareado'
            pvalue = stats.ttest_rel(x, y).pvalue
        else:
            test_name = 'Wilcoxon'
            pvalue = stats.wilcoxon(x, y).pvalue
        resultados.append({
            'variable': nombre,
            'n_pares': len(tabla),
            'mediana_qaoa': np.median(x),
            'mediana_qaoa_warmstart': np.median(y),
            'shapiro_p_diferencias': shapiro_p,
            'prueba': test_name,
            'p_valor': pvalue,
            'cohens_d_paired': cohens_d_paired(x, y)
        })

    display(pd.DataFrame(resultados))


## Correlaciones entre complejidad y coste computacional

La matriz de correlaciones resume la relación entre tamaño, penalización, tiempo y métricas internas. En AP debe interpretarse con cautela porque solo se incluyen dos tamaños (`2x2` y `3x3`) y, por ahora, el número de filas válidas es reducido.


In [ ]:
cols_corr = [
    'num_filas', 'num_columnas', 'num_variables_qubo', 'optimo', 'penalizacion',
    'tiempo_total', 't_solver', 't_cuantico', 't_clasico',
    'prob_optimo', 'ratio_optimo_medio', 'prob_optimo_muestras',
    'prob_optimo_starts', 'tasa_factibilidad', 'ratio_medio_factibles',
    'coste_medio_start', 'std_coste_start', 'tiempo_medio_start'
]
cols_corr = [c for c in cols_corr if c in df.columns and df[c].notna().any()]
corr = df[cols_corr].corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Matriz de correlación entre complejidad, tiempo y calidad interna')
plt.tight_layout()
plt.show()


## Conclusión provisional

Con los resultados AP válidos disponibles (`2x2` y `3x3`), SA encuentra siempre una solución final factible y óptima. La diferencia más visible entre tamaños no está en la calidad final, sino en el incremento del tiempo y en el deterioro de las métricas internas de muestreo al pasar a `3x3`.

No se incluyen conclusiones sobre `4x4`, porque esas instancias dieron error y quedan fuera del análisis. Tampoco debe compararse QAOA contra SA hasta disponer de filas AP válidas con el esquema actual para `qaoa` y `qaoa_warmstart`.
